In [ ]:
import os
import sys
import time
import json
import logging
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib as mpl
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.core.repair import Repair

warnings.filterwarnings("ignore")

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["mathtext.fontset"] = "dejavusans"
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
logging.getLogger("matplotlib").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Glyph.*missing from current font.*")
warnings.filterwarnings("ignore", message=".*Font 'default' does not have a glyph.*")


class EnsembleModel:
    """封装 XGBoost 和 RandomForest 的加权融合代理模型。"""

    def __init__(self, xgb_model, rf_model, scaler_X, weight_xgb):
        self.xgb_model = xgb_model
        self.rf_model = rf_model
        self.scaler_X = scaler_X
        self.weight_xgb = weight_xgb
        self.is_fitted_ = True

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values
        elif isinstance(X, pd.Series):
            X = X.to_frame().T.values
        elif isinstance(X, np.ndarray):
            if X.ndim == 1:
                X = X.reshape(1, -1)
        else:
            X = np.asarray(X)
            if X.ndim == 1:
                X = X.reshape(1, -1)

        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X_scaled = self.scaler_X.transform(X)
        pred_xgb = self.xgb_model.predict(X_scaled)
        pred_rf = self.rf_model.predict(X)
        return np.ravel(self.weight_xgb * pred_xgb + (1 - self.weight_xgb) * pred_rf)


def get_default_feature_bounds():
    return {
        "MPDC": {"bounds": [0.5, 5.0], "is_int": False},
        "TMCC": {"bounds": [0.1, 1.0], "is_int": False},
        "MPDIPT": {"bounds": [1.0, 5.0], "is_int": False},
        "TMCIPT": {"bounds": [1.0, 3.0], "is_int": False},
        "MT": {"bounds": [0.0, 22.0], "is_int": True},
        "PTE": {"bounds": [45.0, 95.0], "is_int": False},
        "PTI": {"bounds": [1.0, 60.0], "is_int": False},
        "OPS": {"bounds": [0.0, 5.0], "is_int": False},
        "CT": {"bounds": [100.0, 250.0], "is_int": False},
        "WCA": {"bounds": [30.0, 115.0], "is_int": False},
        "WP": {"bounds": [0.5, 22.5], "is_int": False},
        "MO": {"bounds": [0.0, 1.0], "is_int": True},
        "DST": {"bounds": [0.0, 5.0], "is_int": True},
        "DSC": {"bounds": [0.5, 4.0], "is_int": False},
        "ST": {"bounds": [22.0, 30.0], "is_int": False},
        "CV": {"bounds": [33.0, 510.0], "is_int": False},
    }


def load_feature_bounds(feature_bounds_path, defaults, expected_features=None):
    merged = {name: {"bounds": list(cfg["bounds"]), "is_int": bool(cfg["is_int"])} for name, cfg in defaults.items()}
    loaded_raw = {}

    if os.path.exists(feature_bounds_path):
        with open(feature_bounds_path, "r", encoding="utf-8") as f:
            loaded_raw = json.load(f)

        for name, value in loaded_raw.items():
            if name not in merged:
                continue

            if isinstance(value, dict):
                bounds = value.get("bounds", merged[name]["bounds"])
                is_int = value.get("is_int", merged[name]["is_int"])
            else:
                bounds = value
                is_int = merged[name]["is_int"]

            if not isinstance(bounds, (list, tuple)) or len(bounds) != 2:
                raise ValueError(f"{name} 的边界格式非法: {bounds}")

            lower, upper = float(bounds[0]), float(bounds[1])
            if lower >= upper:
                raise ValueError(f"{name} 的边界非法: [{lower}, {upper}]")

            merged[name] = {"bounds": [lower, upper], "is_int": bool(is_int)}

    if expected_features is not None and len(merged) != expected_features:
        raise ValueError(f"特征数与模型期望不一致: {len(merged)} vs {expected_features}")

    with open(feature_bounds_path, "w", encoding="utf-8") as f:
        json.dump(merged, f, ensure_ascii=False, indent=4)

    return merged, loaded_raw


print("\n" + "=" * 70)
print("阶段 1/5: 系统初始化、代理模型加载与混合整数空间定义")
print("=" * 70)

model_dir = "saved_models"
model_paths = {
    "Jw (水通量)": os.path.join(model_dir, "model_Jw.pkl"),
    "Js (盐通量)": os.path.join(model_dir, "model_Js.pkl"),
    "R  (截留率)": os.path.join(model_dir, "model_R.pkl"),
}

for name, path in model_paths.items():
    if not os.path.exists(path):
        print(f"致命错误: 找不到模型文件 '{path}'")
        sys.exit(1)

print("正在加载 XGBoost + RandomForest 集成模型...")
load_start = time.time()
model_Jw = joblib.load(model_paths["Jw (水通量)"])
model_Js = joblib.load(model_paths["Js (盐通量)"])
model_R = joblib.load(model_paths["R  (截留率)"])
print(f"模型加载完成，耗时 {time.time() - load_start:.3f} 秒")

try:
    expected_features = model_Jw.scaler_X.n_features_in_
    print(f"模型期望输入维度: {expected_features}")
except AttributeError:
    expected_features = None
    print("无法自动探测模型输入维度，将使用边界配置校验。")

feature_bounds_path = os.path.join(model_dir, "feature_bounds.json")
feature_bounds_default = get_default_feature_bounds()
feature_bounds_config, raw_feature_bounds = load_feature_bounds(
    feature_bounds_path,
    feature_bounds_default,
    expected_features=expected_features,
)

feature_names = list(feature_bounds_config.keys())
n_features = len(feature_names)
xl = np.array([cfg["bounds"][0] for cfg in feature_bounds_config.values()], dtype=float)
xu = np.array([cfg["bounds"][1] for cfg in feature_bounds_config.values()], dtype=float)
is_integer = np.array([cfg["is_int"] for cfg in feature_bounds_config.values()], dtype=bool)
int_var_indices = np.where(is_integer)[0].tolist()
int_var_names = [feature_names[i] for i in int_var_indices]


class IntegerVariableRepair(Repair):
    def __init__(self, int_indices):
        super().__init__()
        self.int_indices = int_indices

    def _do(self, problem, X, **kwargs):
        if self.int_indices:
            X[:, self.int_indices] = np.round(X[:, self.int_indices])
        return X


integer_repair = IntegerVariableRepair(int_var_indices)

if expected_features is not None and n_features != expected_features:
    print(f"致命错误: 边界定义特征数 {n_features} 与模型期望 {expected_features} 不一致")
    sys.exit(1)

if np.any(xl >= xu):
    print("致命错误: 存在下界大于等于上界的变量")
    sys.exit(1)

print("\n搜索空间配置报告:")
print(f"  - 优化变量总数: {n_features}")
print(f"  - 连续变量数量: {n_features - len(int_var_indices)}")
print(f"  - 整数变量数量: {len(int_var_indices)} -> {int_var_names}")
print(f"  - 搜索空间体积: {np.prod(xu - xl):.2e}")

df_bounds = pd.DataFrame(
    {
        "Feature Name": feature_names,
        "Type": ["Integer" if flag else "Continuous" for flag in is_integer],
        "Lower Bound": xl,
        "Upper Bound": xu,
        "Range (Δ)": xu - xl,
    }
)
with pd.option_context("display.max_rows", None, "display.float_format", "{:.2f}".format):
    print(df_bounds.to_string(index=False))

print("\n执行模型冒烟测试...")
np.random.seed(42)
test_input = np.random.uniform(xl, xu, size=(5, n_features))
test_input[:, int_var_indices] = np.round(test_input[:, int_var_indices])

try:
    pred_jw = model_Jw.predict(test_input)
    pred_js = model_Js.predict(test_input)
    pred_r = model_R.predict(test_input)
    print(f"  - Jw 预测范围: [{np.min(pred_jw):.2f}, {np.max(pred_jw):.2f}] LMH")
    print(f"  - Js 预测范围: [{np.min(pred_js):.2f}, {np.max(pred_js):.2f}] gMH")
    print(f"  - R  预测范围: [{np.min(pred_r):.2f}, {np.max(pred_r):.2f}] %")
    assert np.all(test_input[:, int_var_indices] == np.round(test_input[:, int_var_indices]))
    print("冒烟测试通过，优化准备完成。")
except Exception as e:
    print(f"冒烟测试失败: {e}")
    sys.exit(1)

print("=" * 70 + "\n")


In [ ]:
constraint_names = ["R>=90%", "Jw>=10", "Js>=0", "R<=100%", "Jw>=0.1", "Js<=100"]


class FOMembraneOptimization(Problem):
    def __init__(self, model_jw, model_js, model_r, xl, xu, int_var_indices=None):
        super().__init__(n_var=len(xl), n_obj=3, n_constr=len(constraint_names), xl=xl, xu=xu)
        self.model_jw = model_jw
        self.model_js = model_js
        self.model_r = model_r
        self.int_var_indices = int_var_indices if int_var_indices is not None else []

    def _evaluate(self, X, out, *args, **kwargs):
        if self.int_var_indices:
            X[:, self.int_var_indices] = np.round(X[:, self.int_var_indices])

        X_clipped = np.clip(X, self.xl, self.xu)
        X_clean = np.nan_to_num(X_clipped, nan=0.0, posinf=self.xu, neginf=self.xl)

        Jw_pred = self.model_jw.predict(X_clean)
        Js_pred = self.model_js.predict(X_clean)
        R_pred = self.model_r.predict(X_clean)

        f1 = -Jw_pred
        f2 = Js_pred
        f3 = -R_pred

        g1 = 90.0 - R_pred
        g2 = 10.0 - Jw_pred
        g3 = -Js_pred
        g4 = R_pred - 100.0
        g5 = 0.1 - Jw_pred
        g6 = Js_pred - 100.0

        out["F"] = np.column_stack([f1, f2, f3])
        out["G"] = np.column_stack([g1, g2, g3, g4, g5, g6])


problem = FOMembraneOptimization(
    model_Jw,
    model_Js,
    model_R,
    xl,
    xu,
    int_var_indices=int_var_indices,
)

print("FO 膜优化问题已定义")
print(f"  - 优化变量: {problem.n_var} 维 (整数变量 {len(int_var_indices)} 个: {int_var_names})")
print("  - 目标函数: 3 个 (Max Jw, Min Js, Max R)")
print(f"  - 约束条件: {problem.n_constr} 个 -> {constraint_names}")


In [ ]:
import time
from pymoo.core.callback import Callback
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

mpl.rcParams["axes.linewidth"] = 1.2
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.size"] = 11


class ConvergenceMonitor(Callback):
    def __init__(self, int_var_indices=None):
        super().__init__()
        self.n_nds_history = []
        self.feasible_ratio_history = []
        self.integer_deviation_history = []
        self.int_var_indices = int_var_indices or []

    def notify(self, algorithm):
        self.n_nds_history.append(len(algorithm.opt))
        cv = algorithm.pop.get("CV")
        feasible_ratio = np.sum(cv <= 0) / len(cv) if cv is not None else 0.0
        self.feasible_ratio_history.append(feasible_ratio)

        if self.int_var_indices:
            pop_vars = algorithm.pop.get("X")
            deviations = []
            for idx in self.int_var_indices:
                fractional_parts = pop_vars[:, idx] - np.round(pop_vars[:, idx])
                deviations.append(np.std(fractional_parts))
            self.integer_deviation_history.append(float(np.mean(deviations)))


def create_nsga2_algorithm(pop_size, n_offsprings):
    return NSGA2(
        pop_size=pop_size,
        n_offsprings=n_offsprings,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=15),
        mutation=PM(prob=1.0 / n_features, eta=20),
        eliminate_duplicates=True,
        repair=integer_repair,
    )


def ensure_2d_array(array):
    array = np.asarray(array)
    return array.reshape(1, -1) if array.ndim == 1 else array


def compute_ensemble_disagreement(model, X):
    X = ensure_2d_array(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X_scaled = model.scaler_X.transform(X)
    pred_xgb = np.ravel(model.xgb_model.predict(X_scaled))
    pred_rf = np.ravel(model.rf_model.predict(X))
    return np.abs(pred_xgb - pred_rf), pred_xgb, pred_rf


def evaluate_objectives_from_models(X):
    X = ensure_2d_array(X).astype(float)
    if int_var_indices:
        X[:, int_var_indices] = np.round(X[:, int_var_indices])
    X = np.clip(X, xl, xu)

    jw = np.ravel(model_Jw.predict(X))
    js = np.ravel(model_Js.predict(X))
    r = np.ravel(model_R.predict(X))
    return X, jw, js, r


def extract_pareto_dataframe(result):
    if result.X is None or len(result.X) == 0:
        raise ValueError("优化结果为空，未找到 Pareto 解。")

    pareto_X_raw = ensure_2d_array(result.X).astype(float)
    pareto_X, pareto_Jw, pareto_Js, pareto_R = evaluate_objectives_from_models(pareto_X_raw)

    jw_disagreement, _, _ = compute_ensemble_disagreement(model_Jw, pareto_X)
    js_disagreement, _, _ = compute_ensemble_disagreement(model_Js, pareto_X)
    r_disagreement, _, _ = compute_ensemble_disagreement(model_R, pareto_X)

    df = pd.DataFrame(pareto_X, columns=feature_names)
    for name in int_var_names:
        df[name] = df[name].round().astype(int)

    df.insert(0, "Pareto_Index", np.arange(len(df), dtype=int))
    df["Jw (LMH)"] = pareto_Jw
    df["Js (gMH)"] = pareto_Js
    df["R (%)"] = pareto_R
    df["Selectivity (Js/Jw g/L)"] = (pareto_Js / np.maximum(pareto_Jw, 1e-10)) * 1000
    df["Js/Jw (g/L)"] = df["Selectivity (Js/Jw g/L)"]
    df["Jw_Model_Disagreement"] = jw_disagreement
    df["Js_Model_Disagreement"] = js_disagreement
    df["R_Model_Disagreement"] = r_disagreement
    df["Mean_Model_Disagreement"] = np.mean(
        np.column_stack([jw_disagreement, js_disagreement, r_disagreement]),
        axis=1,
    )
    return pareto_X, pareto_Jw, pareto_Js, pareto_R, df


def build_boundary_summary(df, feature_bounds_cfg, proximity_ratio=0.05):
    records = []
    for feature_name, cfg in feature_bounds_cfg.items():
        lower, upper = cfg["bounds"]
        feature_values = df[feature_name].to_numpy(dtype=float)
        span = upper - lower
        tol = max(span * proximity_ratio, 1e-8)
        lower_hits = int(np.sum(np.abs(feature_values - lower) <= tol))
        upper_hits = int(np.sum(np.abs(feature_values - upper) <= tol))
        records.append(
            {
                "Feature": feature_name,
                "Lower_Bound": lower,
                "Upper_Bound": upper,
                "Near_Lower_Count": lower_hits,
                "Near_Upper_Count": upper_hits,
                "Near_Lower_Ratio": lower_hits / max(len(df), 1),
                "Near_Upper_Ratio": upper_hits / max(len(df), 1),
                "Observed_Min": float(np.min(feature_values)),
                "Observed_Max": float(np.max(feature_values)),
                "Is_Integer": bool(cfg["is_int"]),
            }
        )
    return pd.DataFrame(records)


def get_non_dominated_mask(F):
    F = np.asarray(F, dtype=float)
    n = F.shape[0]
    mask = np.ones(n, dtype=bool)
    for i in range(n):
        dominated_by_others = np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)
        dominated_by_others[i] = False
        if np.any(dominated_by_others):
            mask[i] = False
    return mask


def run_optimization(seed, pop_size, n_offsprings, n_gen, verbose=False):
    algorithm = create_nsga2_algorithm(pop_size, n_offsprings)
    monitor = ConvergenceMonitor(int_var_indices=int_var_indices)
    start_time = time.time()
    result = minimize(
        problem,
        algorithm,
        termination=("n_gen", n_gen),
        seed=seed,
        callback=monitor,
        verbose=verbose,
    )
    elapsed = time.time() - start_time
    return result, monitor, elapsed


POP_SIZE = 10000
N_OFFSPRINGS = 200
N_GEN = 500
DEFAULT_SEED = 42

print("\n" + "=" * 70)
print("阶段 3/5: 启动 NSGA-II 混合整数多目标优化")
print("=" * 70)
print(f"优化目标: 3 个")
print(f"约束条件: {problem.n_constr} 个 -> {constraint_names}")
print(f"混合变量: {n_features} 维 (连续 {n_features - len(int_var_indices)} / 整数 {len(int_var_indices)})")
print(f"种群规模: {POP_SIZE} | 每代后代: {N_OFFSPRINGS}")
print(f"最大迭代数: {N_GEN} 代")
print("=" * 70 + "\n")

res, monitor, optimization_seconds = run_optimization(
    seed=DEFAULT_SEED,
    pop_size=POP_SIZE,
    n_offsprings=N_OFFSPRINGS,
    n_gen=N_GEN,
    verbose=True,
)

if res.X is None or len(res.X) == 0:
    print("未找到任何满足约束条件的 Pareto 最优解，请检查约束条件设置。")
    sys.exit(1)

pareto_X, pareto_Jw, pareto_Js, pareto_R, df_pareto = extract_pareto_dataframe(res)
pareto_F = np.column_stack([-pareto_Jw, pareto_Js, -pareto_R])

print("\n" + "=" * 70)
print("优化任务完成，结果摘要如下")
print("=" * 70)
print(f"总耗时: {optimization_seconds:.2f} 秒")
print(f"Pareto 解数量: {len(df_pareto)}")
print(f"Jw 范围: [{pareto_Jw.min():.2f}, {pareto_Jw.max():.2f}]")
print(f"Js 范围: [{pareto_Js.min():.2f}, {pareto_Js.max():.2f}]")
print(f"R 范围 : [{pareto_R.min():.2f}, {pareto_R.max():.2f}]")

constraint_matrix = np.column_stack(
    [
        90.0 - pareto_R,
        10.0 - pareto_Jw,
        -pareto_Js,
        pareto_R - 100.0,
        0.1 - pareto_Jw,
        pareto_Js - 100.0,
    ]
)
constraint_violations = np.sum(constraint_matrix > 1e-6, axis=0)
print("\n约束违反统计:")
for name, violations in zip(constraint_names, constraint_violations):
    print(f"  - {name}: {violations}")

df_pareto = df_pareto.sort_values(
    by=["Jw (LMH)", "R (%)", "Js (gMH)"],
    ascending=[False, False, True],
).reset_index(drop=True)
df_pareto["Pareto_Index"] = np.arange(len(df_pareto), dtype=int)
df_pareto.to_csv("pareto_optimal_solutions.csv", index=False, encoding="utf-8-sig")

boundary_summary = build_boundary_summary(df_pareto, feature_bounds_config)
boundary_summary.to_csv("boundary_hit_summary.csv", index=False, encoding="utf-8-sig")

table_s1_search_space = pd.DataFrame(
    {
        "Feature": feature_names,
        "Variable_Type": ["Integer" if flag else "Continuous" for flag in is_integer],
        "Lower_Bound": xl,
        "Upper_Bound": xu,
        "Search_Range": xu - xl,
    }
)
table_s1_search_space.to_csv("Table_S1_Search_Space_Definition.csv", index=False, encoding="utf-8-sig")

table_s2_constraints = pd.DataFrame(
    {
        "Constraint_ID": [f"C{i + 1}" for i in range(len(constraint_names))],
        "Constraint_Name": constraint_names,
        "Mathematical_Definition": [
            "R >= 90",
            "Jw >= 10",
            "Js >= 0",
            "R <= 100",
            "Jw >= 0.1",
            "Js <= 100",
        ],
        "Constraint_Type": ["Primary", "Primary", "Primary", "Primary", "Physical", "Physical"],
    }
)
table_s2_constraints.to_csv("Table_S2_Constraint_Definitions.csv", index=False, encoding="utf-8-sig")

table_s3_run_summary = pd.DataFrame(
    [
        {"Metric": "Model family", "Value": "XGBoost + RandomForest"},
        {"Metric": "Optimization algorithm", "Value": "NSGA-II"},
        {"Metric": "Random seed", "Value": DEFAULT_SEED},
        {"Metric": "Population size", "Value": POP_SIZE},
        {"Metric": "Offspring per generation", "Value": N_OFFSPRINGS},
        {"Metric": "Number of generations", "Value": N_GEN},
        {"Metric": "Number of variables", "Value": n_features},
        {"Metric": "Number of integer variables", "Value": len(int_var_indices)},
        {"Metric": "Number of Pareto solutions", "Value": len(df_pareto)},
        {"Metric": "Optimization time (s)", "Value": round(float(optimization_seconds), 3)},
        {"Metric": "Jw min (LMH)", "Value": round(float(pareto_Jw.min()), 4)},
        {"Metric": "Jw max (LMH)", "Value": round(float(pareto_Jw.max()), 4)},
        {"Metric": "Js min (gMH)", "Value": round(float(pareto_Js.min()), 4)},
        {"Metric": "Js max (gMH)", "Value": round(float(pareto_Js.max()), 4)},
        {"Metric": "R min (%)", "Value": round(float(pareto_R.min()), 4)},
        {"Metric": "R max (%)", "Value": round(float(pareto_R.max()), 4)},
        {"Metric": "Mean model disagreement", "Value": round(float(df_pareto["Mean_Model_Disagreement"].mean()), 6)},
    ]
)
table_s3_run_summary.to_csv("Table_S3_Optimization_Run_Summary.csv", index=False, encoding="utf-8-sig")

table_s4_boundary_activity = boundary_summary.rename(
    columns={
        "Feature": "Feature",
        "Near_Lower_Count": "Near_Lower_Bound_Count",
        "Near_Upper_Count": "Near_Upper_Bound_Count",
        "Near_Lower_Ratio": "Near_Lower_Bound_Ratio",
        "Near_Upper_Ratio": "Near_Upper_Bound_Ratio",
    }
)
table_s4_boundary_activity.to_csv("Table_S4_Boundary_Activity.csv", index=False, encoding="utf-8-sig")

paper_figure_manifest = pd.DataFrame(
    [
        {
            "Figure_File": "Fig1_Pareto_Front_3D.png",
            "Suggested_Title": "Three-dimensional Pareto front generated by NSGA-II",
            "Suggested_Caption": "Three-dimensional Pareto front of water flux, salt flux, and salt rejection predicted by the XGBoost + RandomForest surrogate models.",
        },
        {
            "Figure_File": "Fig2_NSGA2_Convergence_MINLP.png",
            "Suggested_Title": "Convergence behavior of the NSGA-II optimization",
            "Suggested_Caption": "Evolution of Pareto front size, feasible-ratio trajectory, and integer-variable stability during mixed-integer NSGA-II optimization.",
        },
        {
            "Figure_File": "Fig3_Pareto_2D_Projections.png",
            "Suggested_Title": "Pairwise trade-off projections of the Pareto front",
            "Suggested_Caption": "Two-dimensional trade-off views highlighting permeability-selectivity and leakage-rejection relationships across the Pareto-optimal set.",
        },
        {
            "Figure_File": "Fig4_Parallel_Coordinates.png",
            "Suggested_Title": "Parallel-coordinate comparison of Pareto-optimal design variables",
            "Suggested_Caption": "Normalized design-variable distributions across Pareto-optimal membrane recipes grouped by water-flux level.",
        },
        {
            "Figure_File": "Fig5_Best_Solution_Analysis.png",
            "Suggested_Title": "Multi-criteria analysis of the recommended membrane recipe",
            "Suggested_Caption": "Radar-chart comparison of the top-ranked solutions and normalized parameter levels of the final recommended recipe.",
        },
        {
            "Figure_File": "Fig6_OAT_Sensitivity_Curves.png",
            "Suggested_Title": "One-at-a-time sensitivity curves around the selected baseline",
            "Suggested_Caption": "Relative percentage changes in water flux, salt flux, and rejection when each design variable is perturbed independently around the selected baseline recipe.",
        },
        {
            "Figure_File": "Fig7_Quantitative_Sensitivity_Heatmap.png",
            "Suggested_Title": "Quantitative sensitivity ranking across membrane-design variables",
            "Suggested_Caption": "Heatmap and local-slope analysis summarizing global and local sensitivities of the surrogate predictions to each design variable.",
        },
    ]
)
paper_figure_manifest.to_csv("paper_figure_manifest.csv", index=False, encoding="utf-8-sig")

RUN_STABILITY_ANALYSIS = True
STABILITY_SEEDS = [7, 21, 42]
STABILITY_POP_SIZE = min(2000, POP_SIZE)
STABILITY_N_OFFSPRINGS = min(200, N_OFFSPRINGS)
STABILITY_N_GEN = min(80, N_GEN)

stability_records = []
stability_fronts = []

if RUN_STABILITY_ANALYSIS:
    print("\n开始多随机种子稳定性分析...")
    for seed in STABILITY_SEEDS:
        sub_res, _, elapsed_seconds = run_optimization(
            seed=seed,
            pop_size=STABILITY_POP_SIZE,
            n_offsprings=STABILITY_N_OFFSPRINGS,
            n_gen=STABILITY_N_GEN,
            verbose=False,
        )
        if sub_res.X is None or len(sub_res.X) == 0:
            stability_records.append(
                {
                    "Seed": seed,
                    "Elapsed_Seconds": elapsed_seconds,
                    "Pareto_Count": 0,
                    "Feasible_Ratio": 0.0,
                    "Mean_Jw": np.nan,
                    "Min_Js": np.nan,
                    "Max_R": np.nan,
                    "Mean_Model_Disagreement": np.nan,
                    "Hypervolume": np.nan,
                    "IGD_Empirical": np.nan,
                }
            )
            continue

        _, jw_seed, js_seed, r_seed, df_seed = extract_pareto_dataframe(sub_res)
        F_seed = np.column_stack([-jw_seed, js_seed, -r_seed])
        stability_fronts.append((seed, F_seed))

        cv_seed = np.column_stack(
            [
                90.0 - r_seed,
                10.0 - jw_seed,
                -js_seed,
                r_seed - 100.0,
                0.1 - jw_seed,
                js_seed - 100.0,
            ]
        )
        feasible_ratio = float(np.mean(np.all(cv_seed <= 1e-6, axis=1)))
        stability_records.append(
            {
                "Seed": seed,
                "Elapsed_Seconds": elapsed_seconds,
                "Pareto_Count": len(df_seed),
                "Feasible_Ratio": feasible_ratio,
                "Mean_Jw": float(np.mean(jw_seed)),
                "Min_Js": float(np.min(js_seed)),
                "Max_R": float(np.max(r_seed)),
                "Mean_Model_Disagreement": float(df_seed["Mean_Model_Disagreement"].mean()),
                "Hypervolume": np.nan,
                "IGD_Empirical": np.nan,
            }
        )

    if stability_fronts:
        combined_front = np.vstack([front for _, front in stability_fronts])
        reference_mask = get_non_dominated_mask(combined_front)
        empirical_reference_front = combined_front[reference_mask]

        ref_point = np.max(combined_front, axis=0) + np.array([1.0, 1.0, 1.0])
        hv_indicator = HV(ref_point=ref_point)
        igd_indicator = IGD(empirical_reference_front)

        hv_igd_by_seed = {}
        for seed, front in stability_fronts:
            hv_igd_by_seed[seed] = {
                "Hypervolume": float(hv_indicator(front)),
                "IGD_Empirical": float(igd_indicator(front)),
            }

        for record in stability_records:
            if record["Seed"] in hv_igd_by_seed:
                record.update(hv_igd_by_seed[record["Seed"]])

    df_stability = pd.DataFrame(stability_records)
    df_stability.to_csv("optimization_seed_stability_summary.csv", index=False, encoding="utf-8-sig")
    df_stability.to_csv("Table_S5_Seed_Stability.csv", index=False, encoding="utf-8-sig")
else:
    df_stability = pd.DataFrame()

analysis_metadata = {
    "model_family": "XGBoost + RandomForest",
    "feature_names": feature_names,
    "integer_variables": int_var_names,
    "feature_bounds": feature_bounds_config,
    "main_run": {
        "seed": DEFAULT_SEED,
        "pop_size": POP_SIZE,
        "n_offsprings": N_OFFSPRINGS,
        "n_gen": N_GEN,
        "elapsed_seconds": optimization_seconds,
        "pareto_count": int(len(df_pareto)),
    },
    "exports": {
        "pareto": "pareto_optimal_solutions.csv",
        "boundary_summary": "boundary_hit_summary.csv",
        "stability_summary": "optimization_seed_stability_summary.csv" if RUN_STABILITY_ANALYSIS else None,
        "convergence_figure": "Fig2_NSGA2_Convergence_MINLP.png",
    },
    "stability_analysis": {
        "enabled": RUN_STABILITY_ANALYSIS,
        "seeds": STABILITY_SEEDS,
        "pop_size": STABILITY_POP_SIZE,
        "n_offsprings": STABILITY_N_OFFSPRINGS,
        "n_gen": STABILITY_N_GEN,
    },
}

analysis_metadata["paper_exports"] = {
    "figure_manifest": "paper_figure_manifest.csv",
    "tables": [
        "Table_S1_Search_Space_Definition.csv",
        "Table_S2_Constraint_Definitions.csv",
        "Table_S3_Optimization_Run_Summary.csv",
        "Table_S4_Boundary_Activity.csv",
        "Table_S5_Seed_Stability.csv" if RUN_STABILITY_ANALYSIS else None,
    ],
}

with open("analysis_metadata.json", "w", encoding="utf-8") as f:
    json.dump(analysis_metadata, f, ensure_ascii=False, indent=4)

if len(monitor.n_nds_history) > 1:
    fig, (ax1, ax3) = plt.subplots(
        2,
        1,
        figsize=(10, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 2]},
    )

    color_blue = "1f77b4"
    color_red = "d62728"

    ax1.set_title("NSGA-II Convergence Monitoring (MINLP)", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Pareto Front Size", color=color_blue, fontsize=12)
    ax1.plot(
        monitor.n_nds_history,
        color=color_blue,
        lw=2.5,
        marker="o",
        markersize=4,
        label=f"Pareto Solutions (Final: {monitor.n_nds_history[-1]})",
    )
    ax1.tick_params(axis="y", labelcolor=color_blue)
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend(loc="upper left", fontsize=10)

    ax2 = ax1.twinx()
    ax2.set_ylabel("Feasible Ratio", color=color_red, fontsize=12)
    ax2.plot(
        monitor.feasible_ratio_history,
        color=color_red,
        lw=2.5,
        linestyle="--",
        label=f"Feasible Ratio (Final: {monitor.feasible_ratio_history[-1]:.2f})",
    )
    ax2.tick_params(axis="y", labelcolor=color_red)
    ax2.set_ylim(0, 1.05)
    ax2.legend(loc="upper right", fontsize=10)

    if monitor.integer_deviation_history:
        dev_history = np.array(monitor.integer_deviation_history, dtype=float)
        dev_history = np.where(dev_history == 0, 1e-15, dev_history)
        ax3.set_xlabel("Generation", fontsize=12)
        ax3.set_ylabel("Integer Deviation (σ)", color="purple", fontsize=12)
        ax3.plot(
            dev_history,
            color="purple",
            lw=2,
            marker="s",
            markersize=3,
            label=f"Deviation (Final: {dev_history[-1]:.2e})",
        )
        ax3.axhline(y=1e-6, color="gray", linestyle=":", alpha=0.7, label="Tolerance (1e-6)")
        ax3.set_yscale("log")
        ax3.set_ylim(1e-15, 1)
        ax3.grid(True, linestyle="--", alpha=0.6)
        ax3.legend(loc="upper right", fontsize=9)
        ax3.set_title("Integer Variable Stability (Lower is Better)", fontsize=11)

    plt.tight_layout()
    plt.savefig("Fig2_NSGA2_Convergence_MINLP.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("收敛监控图已保存至 'Fig2_NSGA2_Convergence_MINLP.png'")

print("=" * 70)


In [ ]:
from pandas.plotting import parallel_coordinates

print("\n" + "=" * 70)
print("阶段 4/5: Pareto 前沿解析与多维可视化")
print("=" * 70)

if "df_pareto" not in globals():
    if not os.path.exists("pareto_optimal_solutions.csv"):
        print("未找到 Pareto 结果文件，跳过可视化。")
    else:
        df_pareto = pd.read_csv("pareto_optimal_solutions.csv")

if "df_pareto" in globals() and len(df_pareto) > 0:
    pareto_Jw = df_pareto["Jw (LMH)"].to_numpy()
    pareto_Js = df_pareto["Js (gMH)"].to_numpy()
    pareto_R = df_pareto["R (%)"].to_numpy()
    pareto_X = df_pareto[feature_names].to_numpy(dtype=float)
    pareto_F = np.column_stack([-pareto_Jw, pareto_Js, -pareto_R])

    print(f"成功提取 {len(df_pareto)} 个 Pareto 最优解。")

    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(
        pareto_Jw,
        pareto_Js,
        pareto_R,
        c=pareto_R,
        cmap="coolwarm",
        s=80,
        alpha=0.85,
        edgecolors="black",
        linewidth=0.5,
        depthshade=True,
    )
    ax.set_xlabel("Water Flux $J_w$ (LMH)", fontsize=13, labelpad=12)
    ax.set_ylabel("Salt Flux $J_s$ (gMH)", fontsize=13, labelpad=12)
    ax.set_zlabel("Salt Rejection $R$ (%)", fontsize=13, labelpad=12)
    ax.set_title("3D Pareto Optimal Front of FO Membranes", fontsize=16, fontweight="bold", pad=20)
    cbar = fig.colorbar(scatter, ax=ax, fraction=0.025, pad=0.08, shrink=0.85)
    cbar.set_label("Salt Rejection $R$ (%)", fontsize=12, labelpad=10)
    ax.view_init(elev=25, azim=-45)
    plt.tight_layout()
    plt.savefig("Fig1_Pareto_Front_3D.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].scatter(pareto_Jw, pareto_R, c="1f77b4", s=60, edgecolor="black", linewidth=0.5, alpha=0.8)
    axes[0].set_xlabel("Water Flux $J_w$ (LMH)", fontsize=12)
    axes[0].set_ylabel("Salt Rejection $R$ (%)", fontsize=12)
    axes[0].set_title("Permeability vs Selectivity", fontsize=14, fontweight="bold")
    axes[0].grid(True, linestyle="--", alpha=0.6)

    axes[1].scatter(pareto_Jw, pareto_Js, c="ff7f0e", s=60, edgecolor="black", linewidth=0.5, alpha=0.8)
    axes[1].set_xlabel("Water Flux $J_w$ (LMH)", fontsize=12)
    axes[1].set_ylabel("Salt Flux $J_s$ (gMH)", fontsize=12)
    axes[1].set_title("Water Flux vs Salt Leakage", fontsize=14, fontweight="bold")
    axes[1].grid(True, linestyle="--", alpha=0.6)

    axes[2].scatter(pareto_Js, pareto_R, c="2ca02c", s=60, edgecolor="black", linewidth=0.5, alpha=0.8)
    axes[2].set_xlabel("Salt Flux $J_s$ (gMH)", fontsize=12)
    axes[2].set_ylabel("Salt Rejection $R$ (%)", fontsize=12)
    axes[2].set_title("Salt Leakage vs Rejection", fontsize=14, fontweight="bold")
    axes[2].grid(True, linestyle="--", alpha=0.6)

    plt.suptitle("2D Pairwise Trade-off Projections of Pareto Front", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("Fig3_Pareto_2D_Projections.png", dpi=300, bbox_inches="tight")
    plt.show()

    df_parallel = df_pareto[feature_names].copy()
    df_parallel_norm = (df_parallel - df_parallel.min()) / (df_parallel.max() - df_parallel.min() + 1e-9)
    jw_quantiles = pd.qcut(df_pareto["Jw (LMH)"], q=min(3, len(df_pareto)), duplicates="drop")
    if getattr(jw_quantiles, "cat", None) is not None:
        categories = ["Low $J_w$", "Medium $J_w$", "High $J_w$"][: len(jw_quantiles.cat.categories)]
        df_parallel_norm["Performance_Class"] = jw_quantiles.cat.rename_categories(categories)
    else:
        df_parallel_norm["Performance_Class"] = "Pareto Solutions"

    fig, ax = plt.subplots(figsize=(16, 7))
    parallel_coordinates(df_parallel_norm, "Performance_Class", colormap="viridis", alpha=0.4, lw=1.5, ax=ax)
    ax.set_xticks(range(len(feature_names)))
    ax.set_xticklabels(feature_names, rotation=45, ha="right", fontsize=11)
    ax.set_ylabel("Normalized Parameter Value", fontsize=13)
    ax.set_title("Parallel Coordinates of Design Variables", fontsize=16, fontweight="bold", pad=15)
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("Fig4_Parallel_Coordinates.png", dpi=300, bbox_inches="tight")
    plt.show()

    jw_norm = (pareto_Jw - pareto_Jw.min()) / (pareto_Jw.max() - pareto_Jw.min() + 1e-9)
    js_norm = (pareto_Js - pareto_Js.min()) / (pareto_Js.max() - pareto_Js.min() + 1e-9)
    r_norm = (pareto_R - pareto_R.min()) / (pareto_R.max() - pareto_R.min() + 1e-9)
    utopia_distance = np.sqrt((1 - jw_norm) ** 2 + (js_norm) ** 2 + (1 - r_norm) ** 2)
    knee_score = jw_norm + r_norm - js_norm

    representative_map = {
        "Max Jw": int(np.argmax(pareto_Jw)),
        "Min Js": int(np.argmin(pareto_Js)),
        "Max R": int(np.argmax(pareto_R)),
        "Knee Point (Max-Min)": int(np.argmax(knee_score)),
        "Utopia Closest": int(np.argmin(utopia_distance)),
    }

    rep_records = []
    row_to_labels = {}
    for label, row_idx in representative_map.items():
        row_to_labels.setdefault(row_idx, []).append(label)

    for row_idx, labels in row_to_labels.items():
        row = df_pareto.iloc[row_idx].copy()
        row["Solution_Type"] = "; ".join(labels)
        rep_records.append(row)

    df_rep = pd.DataFrame(rep_records)
    leading_cols = ["Solution_Type", "Pareto_Index"] + feature_names
    trailing_cols = [col for col in df_rep.columns if col not in leading_cols]
    df_rep = df_rep[leading_cols + trailing_cols]
    df_rep.to_csv("representative_solutions.csv", index=False, encoding="utf-8-sig")
    df_rep.to_csv("Table_S6_Representative_Solutions.csv", index=False, encoding="utf-8-sig")

    print("\n代表性方案摘要:")
    print(df_rep[["Solution_Type", "Jw (LMH)", "Js (gMH)", "R (%)", "Selectivity (Js/Jw g/L)"]].to_string(index=False))
    print("代表性方案已导出至 'representative_solutions.csv'")

print("=" * 70)


In [ ]:
print("\n" + "=" * 70)
print("阶段 5/5: 多准则智能决策与最佳方案解析")
print("=" * 70)


def min_max_normalize(x, is_cost=False):
    x = np.asarray(x, dtype=float)
    x_min, x_max = np.min(x), np.max(x)
    norm = (x - x_min) / (x_max - x_min + 1e-10)
    return 1 - norm if is_cost else norm


def topsis_method(decision_matrix, weights, is_cost):
    norm_matrix = decision_matrix / np.sqrt((decision_matrix ** 2).sum(axis=0) + 1e-10)
    weighted_matrix = norm_matrix * weights
    ideal_best = np.where(is_cost, weighted_matrix.min(axis=0), weighted_matrix.max(axis=0))
    ideal_worst = np.where(is_cost, weighted_matrix.max(axis=0), weighted_matrix.min(axis=0))
    d_best = np.sqrt(((weighted_matrix - ideal_best) ** 2).sum(axis=1))
    d_worst = np.sqrt(((weighted_matrix - ideal_worst) ** 2).sum(axis=1))
    return d_worst / (d_best + d_worst + 1e-10)


if "df_pareto" not in globals():
    df_pareto = pd.read_csv("pareto_optimal_solutions.csv")

if "Pareto_Index" not in df_pareto.columns:
    df_pareto.insert(0, "Pareto_Index", np.arange(len(df_pareto), dtype=int))

pareto_Jw = df_pareto["Jw (LMH)"].to_numpy(dtype=float)
pareto_Js = df_pareto["Js (gMH)"].to_numpy(dtype=float)
pareto_R = df_pareto["R (%)"].to_numpy(dtype=float)

weights = np.array([0.4, 0.3, 0.3])
is_cost = np.array([False, True, False])

jw_norm_wsm = min_max_normalize(pareto_Jw)
js_norm_wsm = min_max_normalize(pareto_Js, is_cost=True)
r_norm_wsm = min_max_normalize(pareto_R)
wsm_scores = weights[0] * jw_norm_wsm + weights[1] * js_norm_wsm + weights[2] * r_norm_wsm

decision_matrix = np.column_stack([pareto_Jw, pareto_Js, pareto_R])
topsis_scores = topsis_method(decision_matrix, weights, is_cost)

df_decision = df_pareto.copy()
df_decision["WSM_Score"] = wsm_scores
df_decision["TOPSIS_Score"] = topsis_scores
df_decision["Rank_WSM"] = df_decision["WSM_Score"].rank(ascending=False, method="min")
df_decision["Rank_TOPSIS"] = df_decision["TOPSIS_Score"].rank(ascending=False, method="min")
df_decision["Borda_Score"] = df_decision["Rank_WSM"] + df_decision["Rank_TOPSIS"]
df_decision["Final_Rank"] = df_decision["Borda_Score"].rank(method="min", ascending=True).astype(int)

df_decision = df_decision.sort_values(
    by=["Final_Rank", "Borda_Score", "TOPSIS_Score", "WSM_Score", "Jw (LMH)", "R (%)"],
    ascending=[True, True, False, False, False, False],
).reset_index(drop=True)

print("\nTop 5 最佳平衡方案:")
display_cols = [
    "Final_Rank",
    "Pareto_Index",
    "Jw (LMH)",
    "Js (gMH)",
    "R (%)",
    "Selectivity (Js/Jw g/L)",
    "WSM_Score",
    "TOPSIS_Score",
    "Borda_Score",
]
print(df_decision[display_cols].head(5).to_string(index=False))

rank1_solutions = df_decision[df_decision["Final_Rank"] == 1].copy()
if len(rank1_solutions) > 1:
    print(f"检测到 {len(rank1_solutions)} 个并列第一方案，将按 TOPSIS/WSM/Jw/R 继续择优。")
    best_recipe_row = rank1_solutions.sort_values(
        by=["TOPSIS_Score", "WSM_Score", "Jw (LMH)", "R (%)"],
        ascending=[False, False, False, False],
    ).iloc[[0]]
else:
    best_recipe_row = rank1_solutions.iloc[[0]]

best_pareto_index = int(best_recipe_row.iloc[0]["Pareto_Index"])
best_solution_params = best_recipe_row.iloc[0][feature_names].astype(float).to_numpy()
best_Jw = float(best_recipe_row.iloc[0]["Jw (LMH)"])
best_Js = float(best_recipe_row.iloc[0]["Js (gMH)"])
best_R = float(best_recipe_row.iloc[0]["R (%)"])

print("\n最佳推荐方案:")
print(best_recipe_row[display_cols + feature_names].to_string(index=False))

fig = plt.figure(figsize=(16, 6))
ax1 = fig.add_subplot(121, polar=True)
categories = ["Water Flux\n($J_w$)", "Salt Rejection\n($R$)", "Selectivity\n($1-J_s$ norm)"]
N_cats = len(categories)
angles = [n / float(N_cats) * 2 * np.pi for n in range(N_cats)]
angles += angles[:1]

top3 = df_decision.head(min(3, len(df_decision))).copy()
colors_radar = ["e74c3c", "3498db", "2ecc71"]

for i, (_, row) in enumerate(top3.iterrows()):
    pareto_idx = int(row["Pareto_Index"])
    values = [
        min_max_normalize(pareto_Jw)[pareto_idx],
        min_max_normalize(pareto_R)[pareto_idx],
        min_max_normalize(pareto_Js, is_cost=True)[pareto_idx],
    ]
    values = [max(v, 0.1) for v in values]
    values += values[:1]
    ax1.plot(angles, values, "o-", linewidth=2, label=f'Rank {int(row["Final_Rank"])}', color=colors_radar[i])
    ax1.fill(angles, values, alpha=0.15, color=colors_radar[i])

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.set_title("Performance Radar of Top 3 Solutions", fontsize=14, fontweight="bold", pad=20)
ax1.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)

ax2 = fig.add_subplot(122)
best_params_norm = ((best_solution_params - xl) / (xu - xl + 1e-10)) * 100
y_pos = np.arange(n_features)
bars = ax2.barh(y_pos, best_params_norm, color="34495e", alpha=0.8, edgecolor="black")
ax2.set_yticks(y_pos)
ax2.set_yticklabels(feature_names, fontsize=10)
ax2.set_xlabel("Parameter Level (% of Search Bound)", fontsize=12)
ax2.set_title("Design Parameters of Rank 1 Solution", fontsize=14, fontweight="bold")
ax2.invert_yaxis()
ax2.axvline(x=50, color="red", linestyle="--", alpha=0.5, label="Midpoint (50%)")
ax2.legend(loc="lower right", fontsize=9)

for bar, val in zip(bars, best_solution_params):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{val:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("Fig5_Best_Solution_Analysis.png", dpi=300, bbox_inches="tight")
plt.show()

df_decision.to_csv("pareto_solutions_decision_ranked.csv", index=False, encoding="utf-8-sig")

table_1_top5 = df_decision[display_cols + feature_names].head(5).copy()
table_1_top5.to_csv("Table_1_Top5_Ranked_Solutions.csv", index=False, encoding="utf-8-sig")

best_solution_summary = best_recipe_row.copy()
best_solution_summary.insert(0, "Best_Pareto_Index", best_pareto_index)
best_solution_summary.to_csv("best_solution_summary.csv", index=False, encoding="utf-8-sig")

best_recipe = best_recipe_row[feature_names].T
best_recipe.columns = ["Optimal_Value"]
best_recipe.index = best_recipe.index.astype(str)
best_recipe["Lower_Bound"] = pd.Series(xl, index=best_recipe.index)
best_recipe["Upper_Bound"] = pd.Series(xu, index=best_recipe.index)
best_recipe.to_csv("best_recipe_recommendation.csv", encoding="utf-8-sig")

table_s7_best_recipe = best_recipe.reset_index().rename(columns={"index": "Parameter"})
table_s7_best_recipe.to_csv("Table_S7_Best_Recommended_Recipe.csv", index=False, encoding="utf-8-sig")
best_recipe_row.to_csv("Table_S7_Best_Recommended_Performance.csv", index=False, encoding="utf-8-sig")

metadata_path = "analysis_metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, "r", encoding="utf-8") as f:
        analysis_metadata = json.load(f)
else:
    analysis_metadata = {}

analysis_metadata["decision_analysis"] = {
    "weights": {"Jw": 0.4, "Js": 0.3, "R": 0.3},
    "method": "WSM + TOPSIS + Borda",
    "best_pareto_index": best_pareto_index,
    "best_solution": {
        "Jw (LMH)": best_Jw,
        "Js (gMH)": best_Js,
        "R (%)": best_R,
    },
    "exports": {
        "decision_ranked": "pareto_solutions_decision_ranked.csv",
        "best_solution_summary": "best_solution_summary.csv",
        "best_recipe_recommendation": "best_recipe_recommendation.csv",
        "best_solution_figure": "Fig5_Best_Solution_Analysis.png",
        "paper_top5_table": "Table_1_Top5_Ranked_Solutions.csv",
        "paper_best_recipe_table": "Table_S7_Best_Recommended_Recipe.csv",
        "paper_best_performance_table": "Table_S7_Best_Recommended_Performance.csv",
    },
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(analysis_metadata, f, ensure_ascii=False, indent=4)

print("\n决策结果已导出:")
print("  - pareto_solutions_decision_ranked.csv")
print("  - best_solution_summary.csv")
print("  - best_recipe_recommendation.csv")
print("=" * 70)


In [ ]:
import json


def get_default_feature_bounds_for_sensitivity():
    return {
        "MPDC": {"bounds": [0.5, 5.0], "is_int": False},
        "TMCC": {"bounds": [0.1, 1.0], "is_int": False},
        "MPDIPT": {"bounds": [1.0, 5.0], "is_int": False},
        "TMCIPT": {"bounds": [1.0, 3.0], "is_int": False},
        "MT": {"bounds": [0.0, 22.0], "is_int": True},
        "PTE": {"bounds": [45.0, 95.0], "is_int": False},
        "PTI": {"bounds": [1.0, 60.0], "is_int": False},
        "OPS": {"bounds": [0.0, 5.0], "is_int": False},
        "CT": {"bounds": [100.0, 250.0], "is_int": False},
        "WCA": {"bounds": [30.0, 115.0], "is_int": False},
        "WP": {"bounds": [0.5, 22.5], "is_int": False},
        "MO": {"bounds": [0.0, 1.0], "is_int": True},
        "DST": {"bounds": [0.0, 5.0], "is_int": True},
        "DSC": {"bounds": [0.5, 4.0], "is_int": False},
        "ST": {"bounds": [22.0, 30.0], "is_int": False},
        "CV": {"bounds": [33.0, 510.0], "is_int": False},
    }


def load_feature_bounds_for_sensitivity(path, defaults):
    config = {name: {"bounds": list(cfg["bounds"]), "is_int": bool(cfg["is_int"])} for name, cfg in defaults.items()}
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            loaded = json.load(f)
        for name, value in loaded.items():
            if name not in config:
                continue
            bounds = value["bounds"] if isinstance(value, dict) else value
            if isinstance(bounds, (list, tuple)) and len(bounds) == 2:
                config[name]["bounds"] = [float(bounds[0]), float(bounds[1])]
            if isinstance(value, dict) and "is_int" in value:
                config[name]["is_int"] = bool(value["is_int"])
    return config


print("\n" + "=" * 70)
print("阶段 6/6: 单变量敏感性分析 (OAT) 与定量敏感度评估")
print("=" * 70)

if "model_Jw" not in globals():
    print("检测到模型未加载，正在自动初始化...")
    model_dir = "saved_models"
    model_Jw = joblib.load(os.path.join(model_dir, "model_Jw.pkl"))
    model_Js = joblib.load(os.path.join(model_dir, "model_Js.pkl"))
    model_R = joblib.load(os.path.join(model_dir, "model_R.pkl"))

if "feature_names" not in globals() or "xl" not in globals() or "xu" not in globals():
    feature_bounds_config = load_feature_bounds_for_sensitivity(
        os.path.join("saved_models", "feature_bounds.json"),
        get_default_feature_bounds_for_sensitivity(),
    )
    feature_names = list(feature_bounds_config.keys())
    xl = np.array([cfg["bounds"][0] for cfg in feature_bounds_config.values()], dtype=float)
    xu = np.array([cfg["bounds"][1] for cfg in feature_bounds_config.values()], dtype=float)

baseline_source = "workspace_center"
if "best_solution_params" in globals():
    baseline_source = "memory:best_solution_params"
    best_solution_params = np.asarray(best_solution_params, dtype=float)
elif os.path.exists("best_solution_summary.csv"):
    baseline_source = "file:best_solution_summary.csv"
    best_solution_params = pd.read_csv("best_solution_summary.csv").iloc[0][feature_names].to_numpy(dtype=float)
elif os.path.exists("pareto_solutions_decision_ranked.csv"):
    baseline_source = "file:pareto_solutions_decision_ranked.csv"
    ranked_df = pd.read_csv("pareto_solutions_decision_ranked.csv").sort_values(
        by=["Final_Rank", "TOPSIS_Score", "WSM_Score"],
        ascending=[True, False, False],
    )
    best_solution_params = ranked_df.iloc[0][feature_names].to_numpy(dtype=float)
elif os.path.exists("pareto_optimal_solutions.csv"):
    baseline_source = "file:pareto_optimal_solutions.csv"
    pareto_df = pd.read_csv("pareto_optimal_solutions.csv").sort_values(
        by=["Jw (LMH)", "R (%)", "Js (gMH)"],
        ascending=[False, False, True],
    )
    best_solution_params = pareto_df.iloc[0][feature_names].to_numpy(dtype=float)
else:
    best_solution_params = (xl + xu) / 2.0

best_solution_params = np.asarray(best_solution_params, dtype=float)
print(f"敏感性分析基准解来源: {baseline_source}")

n_test_points = 50
n_features = len(feature_names)
n_cols = 4
n_rows = int(np.ceil(n_features / n_cols))
sensitivity_amp = np.zeros((n_features, 3))
sensitivity_slope = np.zeros((n_features, 3))
target_names = ["$J_w$ (Flux)", "$J_s$ (Salt)", "$R$ (Rejection)"]

baseline_preds = np.array(
    [
        model_Jw.predict(best_solution_params.reshape(1, -1))[0],
        model_Js.predict(best_solution_params.reshape(1, -1))[0],
        model_R.predict(best_solution_params.reshape(1, -1))[0],
    ]
)

oat_records = []

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, param_name in enumerate(feature_names):
    ax = axes[i]
    param_min, param_max = xl[i], xu[i]
    test_values = np.linspace(param_min, param_max, n_test_points)

    design_matrix = np.tile(best_solution_params, (n_test_points, 1))
    design_matrix[:, i] = test_values

    Jw_pred = model_Jw.predict(design_matrix)
    Js_pred = model_Js.predict(design_matrix)
    R_pred = model_R.predict(design_matrix)

    Jw_pct = (Jw_pred - baseline_preds[0]) / (np.abs(baseline_preds[0]) + 1e-10) * 100
    Js_pct = (Js_pred - baseline_preds[1]) / (np.abs(baseline_preds[1]) + 1e-10) * 100
    R_pct = (R_pred - baseline_preds[2]) / (np.abs(baseline_preds[2]) + 1e-10) * 100

    oat_records.append(
        pd.DataFrame(
            {
                "Parameter": param_name,
                "Param_Index": i,
                "Test_Value": test_values,
                "Jw_pred_raw": Jw_pred,
                "Js_pred_raw": Js_pred,
                "R_pred_raw": R_pred,
                "Jw_change_pct": Jw_pct,
                "Js_change_pct": Js_pct,
                "R_change_pct": R_pct,
            }
        )
    )

    preds_pct = [Jw_pct, Js_pct, R_pct]
    colors = ["1f77b4", "ff7f0e", "2ca02c"]

    for k, (pct, color, label) in enumerate(zip(preds_pct, colors, target_names)):
        ax.plot(test_values, pct, color=color, lw=2.5, label=label, alpha=0.9)
        sensitivity_amp[i, k] = np.max(pct) - np.min(pct)
        baseline_idx = int(np.argmin(np.abs(test_values - best_solution_params[i])))
        if 0 < baseline_idx < n_test_points - 1:
            dx = test_values[baseline_idx + 1] - test_values[baseline_idx - 1]
            dy = pct[baseline_idx + 1] - pct[baseline_idx - 1]
            sensitivity_slope[i, k] = dy / (dx + 1e-10) if dx != 0 else 0.0

    baseline_val = best_solution_params[i]
    ax.axvline(baseline_val, color="red", linestyle="--", alpha=0.6, lw=1.5, label="Baseline")
    ax.axhline(0, color="black", linestyle="-", alpha=0.3, lw=1)
    ax.set_title(f"{param_name} (Base: {baseline_val:.2f})", fontsize=12, fontweight="bold")
    ax.set_ylabel("Change from Baseline (%)", fontsize=10)
    if i >= (n_rows - 1) * n_cols:
        ax.set_xlabel("Parameter Value", fontsize=10)
    else:
        ax.set_xticklabels([])
    ax.grid(True, linestyle="--", alpha=0.4)
    if i == 0:
        ax.legend(fontsize=9, loc="best", framealpha=0.8)

for j in range(i + 1, n_rows * n_cols):
    axes[j].axis("off")

plt.suptitle("One-at-a-Time (OAT) Sensitivity Analysis: Relative Change from Baseline", fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("Fig6_OAT_Sensitivity_Curves.png", dpi=300, bbox_inches="tight")
plt.show()

df_fig6_raw = pd.concat(oat_records, ignore_index=True)

df_fig6_baseline = pd.DataFrame(
    {
        "Parameter": feature_names,
        "Param_Index": np.arange(n_features),
        "Baseline_Value": best_solution_params,
        "Lower_Bound": xl,
        "Upper_Bound": xu,
        "Baseline_Source": baseline_source,
        "Jw_Baseline_Pred": baseline_preds[0],
        "Js_Baseline_Pred": baseline_preds[1],
        "R_Baseline_Pred": baseline_preds[2],
    }
)

sens_amp_norm = (sensitivity_amp - sensitivity_amp.min(axis=0)) / (
    sensitivity_amp.max(axis=0) - sensitivity_amp.min(axis=0) + 1e-10
)

df_fig7_amp = pd.DataFrame(
    {
        "Parameter": feature_names,
        "Jw_Amplitude_pct": sensitivity_amp[:, 0],
        "Js_Amplitude_pct": sensitivity_amp[:, 1],
        "R_Amplitude_pct": sensitivity_amp[:, 2],
        "Jw_Amplitude_normalized": sens_amp_norm[:, 0],
        "Js_Amplitude_normalized": sens_amp_norm[:, 1],
        "R_Amplitude_normalized": sens_amp_norm[:, 2],
    }
)

df_fig7_slope = pd.DataFrame(
    {
        "Parameter": feature_names,
        "Jw_Local_Slope_pct_per_unit": sensitivity_slope[:, 0],
        "Js_Local_Slope_pct_per_unit": sensitivity_slope[:, 1],
        "R_Local_Slope_pct_per_unit": sensitivity_slope[:, 2],
    }
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), gridspec_kw={"width_ratios": [1.2, 1]})
im1 = ax1.imshow(sens_amp_norm.T, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax1.set_xticks(range(n_features))
ax1.set_xticklabels(feature_names, rotation=45, ha="right", fontsize=11)
ax1.set_yticks(range(3))
ax1.set_yticklabels(target_names, fontsize=12)
ax1.set_title("Global Sensitivity Amplitude\n(Normalized Max-Min % Change)", fontsize=14, fontweight="bold")

for i in range(n_features):
    for j in range(3):
        text_val = f"{sensitivity_amp[i, j]:.1f}%"
        text_color = "white" if sens_amp_norm[i, j] > 0.6 else "black"
        ax1.text(i, j, text_val, ha="center", va="center", color=text_color, fontsize=9)

fig.colorbar(im1, ax=ax1, shrink=0.8, label="Normalized Sensitivity")

x_pos = np.arange(n_features)
width = 0.25
ax2.barh(x_pos - width, sensitivity_slope[:, 0], width, label=target_names[0], color="1f77b4", alpha=0.8)
ax2.barh(x_pos, sensitivity_slope[:, 1], width, label=target_names[1], color="ff7f0e", alpha=0.8)
ax2.barh(x_pos + width, sensitivity_slope[:, 2], width, label=target_names[2], color="2ca02c", alpha=0.8)
ax2.set_yticks(x_pos)
ax2.set_yticklabels(feature_names, fontsize=11)
ax2.set_xlabel("Local Derivative (% Change / Unit Parameter)", fontsize=12)
ax2.set_title("Local Marginal Sensitivity at Baseline\n(Gradient/Slope)", fontsize=14, fontweight="bold")
ax2.axvline(0, color="black", linestyle="-", lw=1)
ax2.legend(loc="lower right", fontsize=10)
ax2.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("Fig7_Quantitative_Sensitivity_Heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

figures_raw_excel_path = "Figures_Raw_Data.xlsx"
raw_data_sheets = [
    ("Fig6_OAT_Curve_Data", df_fig6_raw),
    ("Fig6_Baseline_Values", df_fig6_baseline),
    ("Fig7_Heatmap_Amplitude", df_fig7_amp),
    ("Fig7_Slope_Bars", df_fig7_slope),
]
try:
    with pd.ExcelWriter(figures_raw_excel_path, engine="openpyxl") as writer:
        for sheet_name, frame in raw_data_sheets:
            frame.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"所有绘制图形的原始数据已导出至 Excel: '{figures_raw_excel_path}'")
    print(f"  - Sheet 'Fig6_OAT_Curve_Data'   : {len(df_fig6_raw)} 行（{n_features} 个参数 × {n_test_points} 个采样点）")
    print(f"  - Sheet 'Fig6_Baseline_Values'  : {len(df_fig6_baseline)} 行")
    print(f"  - Sheet 'Fig7_Heatmap_Amplitude': {len(df_fig7_amp)} 行")
    print(f"  - Sheet 'Fig7_Slope_Bars'       : {len(df_fig7_slope)} 行")
except Exception as excel_error:
    print(f"Excel 导出失败（{excel_error}），已改为分别导出 CSV 文件：")
    for sheet_name, frame in raw_data_sheets:
        csv_path = f"{sheet_name}.csv"
        frame.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"  - {csv_path}")

df_sens = pd.DataFrame(
    {
        "Parameter": feature_names,
        "Baseline_Value": best_solution_params,
        "Lower_Bound": xl,
        "Upper_Bound": xu,
        "Baseline_Source": baseline_source,
        "Jw_Amplitude(%)": sensitivity_amp[:, 0],
        "Js_Amplitude(%)": sensitivity_amp[:, 1],
        "R_Amplitude(%)": sensitivity_amp[:, 2],
        "Jw_Local_Slope": sensitivity_slope[:, 0],
        "Js_Local_Slope": sensitivity_slope[:, 1],
        "R_Local_Slope": sensitivity_slope[:, 2],
    }
).sort_values("Jw_Amplitude(%)", ascending=False)
df_sens.to_csv("sensitivity_analysis_report.csv", index=False, encoding="utf-8-sig")

sensitivity_tables = []
for target_col, slope_col, target_name in [
    ("Jw_Amplitude(%)", "Jw_Local_Slope", "Jw"),
    ("Js_Amplitude(%)", "Js_Local_Slope", "Js"),
    ("R_Amplitude(%)", "R_Local_Slope", "R"),
]:
    ranked = df_sens[["Parameter", "Baseline_Value", "Lower_Bound", "Upper_Bound", target_col, slope_col]].copy()
    ranked = ranked.sort_values(target_col, ascending=False).reset_index(drop=True)
    ranked.insert(0, "Target", target_name)
    ranked.insert(1, "Rank", np.arange(1, len(ranked) + 1))
    ranked = ranked.rename(columns={target_col: "Sensitivity_Amplitude(%)", slope_col: "Local_Slope"})
    sensitivity_tables.append(ranked)

table_s8_sensitivity = pd.concat(sensitivity_tables, ignore_index=True)
table_s8_sensitivity.to_csv("Table_S8_Sensitivity_Ranking.csv", index=False, encoding="utf-8-sig")

top_jw = df_sens.sort_values("Jw_Amplitude(%)", ascending=False).head(3)["Parameter"].tolist()
top_js = df_sens.sort_values("Js_Amplitude(%)", ascending=False).head(3)["Parameter"].tolist()
top_r = df_sens.sort_values("R_Amplitude(%)", ascending=False).head(3)["Parameter"].tolist()

summary_lines = [
    " Paper Results Summary",
    "",
    " Optimization Setup",
    f"- Surrogate model: XGBoost + RandomForest ensemble",
    f"- Optimizer: NSGA-II mixed-integer multi-objective optimization",
    f"- Design variables: {n_features} ({len(int_var_indices)} integer variables: {', '.join(int_var_names)})",
    f"- Main-run settings: population size = {POP_SIZE}, offspring = {N_OFFSPRINGS}, generations = {N_GEN}, seed = {DEFAULT_SEED}",
    "",
    " Pareto Front Summary",
    f"- Number of Pareto-optimal solutions: {len(df_pareto)}",
    f"- Water flux range (LMH): {pareto_Jw.min():.4f} to {pareto_Jw.max():.4f}",
    f"- Salt flux range (gMH): {pareto_Js.min():.4f} to {pareto_Js.max():.4f}",
    f"- Rejection range (%): {pareto_R.min():.4f} to {pareto_R.max():.4f}",
    f"- Mean ensemble disagreement: {df_pareto['Mean_Model_Disagreement'].mean():.6f}",
    "",
    " Recommended Solution",
    f"- Selected Pareto index: {best_pareto_index}",
    f"- Water flux (LMH): {best_Jw:.4f}",
    f"- Salt flux (gMH): {best_Js:.4f}",
    f"- Rejection (%): {best_R:.4f}",
    "",
    " Sensitivity Highlights",
    f"- Top-3 variables affecting Jw amplitude: {', '.join(top_jw)}",
    f"- Top-3 variables affecting Js amplitude: {', '.join(top_js)}",
    f"- Top-3 variables affecting R amplitude: {', '.join(top_r)}",
    "",
    " Paper-Ready Exports",
    "- Figures: see paper_figure_manifest.csv",
    "- Figures raw data: Figures_Raw_Data.xlsx",
    "- Supplementary tables: Table_S1 to Table_S8 CSV files",
    "- Main ranking table: Table_1_Top5_Ranked_Solutions.csv",
]

if "df_stability" in globals() and isinstance(df_stability, pd.DataFrame) and not df_stability.empty:
    summary_lines.extend(
        [
            "",
            " Multi-Seed Stability",
            f"- Seeds evaluated: {', '.join(map(str, STABILITY_SEEDS))}",
            f"- Mean Pareto count across seeds: {df_stability['Pareto_Count'].mean():.2f}",
            f"- Mean hypervolume across seeds: {df_stability['Hypervolume'].dropna().mean():.6f}",
            f"- Mean empirical IGD across seeds: {df_stability['IGD_Empirical'].dropna().mean():.6f}",
        ]
    )

with open("paper_results_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines) + "\n")

metadata_path = "analysis_metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, "r", encoding="utf-8") as f:
        analysis_metadata = json.load(f)
else:
    analysis_metadata = {}

analysis_metadata["paper_reporting"] = {
    "summary_markdown": "paper_results_summary.md",
    "sensitivity_table": "Table_S8_Sensitivity_Ranking.csv",
    "main_table": "Table_1_Top5_Ranked_Solutions.csv",
    "figures_raw_data_excel": figures_raw_excel_path,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(analysis_metadata, f, ensure_ascii=False, indent=4)

print("敏感性分析报告已保存至 'sensitivity_analysis_report.csv'")
print("论文摘要与补充表已导出:")
print("  - Table_S8_Sensitivity_Ranking.csv")
print("  - paper_results_summary.md")
print("=" * 70)